# ViLegal_QRHE: Đánh giá mô hình trên tập Test


In [1]:
!pip install torchao transformers torchvision torchaudio

In [1]:
pip install --upgrade torchao  # Chạy đên tránh lỗi phiên bản khi load model

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 7.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
%pip install -U  transformers datasets peft accelerate huggingface_hub safetensors  torch torchvision torchaudio torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 382.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 328.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 499.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 343.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB ? eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 814.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB ? eta 0:00:00
  Attempting uninstall: torchao
    Found existing inst

# Mount drive

In [2]:
IS_COLAB = True

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')



Mounted at /content/drive


# Import lib

In [3]:
import os
import pandas as pd
import json
from pathlib import Path
from dotenv import load_dotenv

import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model
from peft import PeftModel



In [4]:
HF_BASE_MODEL='ntphuc149/ViLegalQwen2.5-1.5B-Base'
HF_MODEL_ID='tmtdev/ViLegalQwen2.5-CitationInference-V1'

MAX_LENGTH = 1024
DATA_DIR = "/content/drive/MyDrive/legal-ai/data"
DATA_PATH = os.path.join(DATA_DIR, "all_extract_citation_data.json")

TASK_ID = 'LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION'


In [5]:

class Prompt:
    SYSTEM_PROMPT = """
    Bạn là model AI xử lý các nhiệm vụ pháp luật Việt Nam.
    Thực hiện đúng TASK được chỉ định trong đầu vào.
    Tuân thủ schema đầu ra của task đó.
    Không tự bịa thông tin hoặc căn cứ pháp luật.
    """

    # TASK: LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION
    LEGAL_QRHE_PROMPT = """
    TASK: LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION

    Nhiệm vụ:
    Trích xuất các cấp trích dẫn và văn bản pháp luật
    được đề cập rõ ràng trong input.

    Chỉ trả về JSON hợp lệ gồm các trường:
    document, part, chapter, section, subsection,
    article, clause, point.

    Cấp không được đề cập phải là null.
    Giá trị trích dẫn là chuỗi.
    Không suy đoán, bịa đặt hoặc bổ sung thông tin.
    Tuân thủ đúng schema đầu ra.
    """

TASK_REGISTRY = {
    "LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION": {
        "prompt": Prompt.LEGAL_QRHE_PROMPT,
        "output_type": "json",   # "json" hoặc "str"
        "max_new_tokens": 256,   # số lượng token có thể sinh ra, tiết kiếm thời gian
    },
}


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(
    HF_BASE_MODEL,
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    dtype=dtype,
)

model = PeftModel.from_pretrained(
    base,
    HF_MODEL_ID,
    is_trainable=False,
)
model.to(device)
model.eval()
model.config.use_cache = True

config.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 73.9MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

In [28]:
def build_messages(task_id: str, input: str) -> list[dict[str, str]]:
    """Tạo template message cho model

    Args:
        task_id (str): Task đã đăng ký giúp model nhận biết để task để thực hiện
        input_text (str): Truy vấn đầu vào để model thực hiện task (có thể là câu hỏi pháp luật, văn bản pháp luật, hoặc ngữ cảnh đã )

    Raises:
        CustomException: Nếu task_id chưa được đăng ký trong tasks

    Returns:
        list[dict[str, str]]: Danh sách các message được tạo ra gồm role và content, giúp model nhận biết task và input_text để thực hiện task
    """

    if task_id not in TASK_REGISTRY:
        raise Exception(f"Task {task_id} is not registered")

    return [
        {"role": "system", "content": Prompt.SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (f"{TASK_REGISTRY[task_id]}\n\n" f"input: {input}"),
        },
    ]

def parse_model_output(raw_output: str, output_type: str = 'str'):
    if output_type == 'str':
        return raw_output

    try:
        decoder = json.JSONDecoder()
        start = raw_output.find("{")

        if start == -1:
            return None

        result, _ = decoder.raw_decode(raw_output[start:])

    except json.JSONDecodeError:
        return None

    return result

from transformers import StoppingCriteria, StoppingCriteriaList


class StopWhenJsonComplete(StoppingCriteria):
    """Dừng generate ngay khi phần model đã sinh chứa trọn 1 object JSON."""

    def __init__(self, tokenizer, prompt_length: int):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        is_json_complete = False

        last_token_text = self.tokenizer.decode(input_ids[0, -1])
        # Chỉ thử parse khi token vừa sinh có dấu "}" (đỡ tốn thời gian)
        if "}" in last_token_text:
            generated_text = self.tokenizer.decode(
                input_ids[0, self.prompt_length:], skip_special_tokens=True
            )
            json_start_position = generated_text.find("{")
            if json_start_position != -1:
                try:
                    json.JSONDecoder().raw_decode(generated_text[json_start_position:])
                    is_json_complete = True
                except json.JSONDecodeError:
                    pass  # JSON chưa đóng xong, sinh tiếp

        return torch.tensor([is_json_complete], dtype=torch.bool, device=input_ids.device)


@torch.inference_mode()
def predict(task_id="LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION", input: str = "", max_new_tokens: int = 1024, return_dict=True) -> dict:

    messages = build_messages(task_id=task_id, input=input)
    output_type = TASK_REGISTRY[task_id]["output_type"]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,          # luôn True, nếu không `**inputs` sẽ lỗi
    ).to(model.device)

    prompt_length = inputs["input_ids"].shape[-1]

    # Qwen dùng <|im_end|> để kết thúc một lượt hội thoại
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    stop_ids = [tokenizer.eos_token_id]
    if im_end_id is not None and im_end_id not in stop_ids:
        stop_ids.append(im_end_id)

    # Chỉ dừng theo JSON khi task trả dict
    stopping_criteria = None
    if return_dict:
        stopping_criteria = StoppingCriteriaList(
            [StopWhenJsonComplete(tokenizer, prompt_length)]
        )

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
        eos_token_id=stop_ids,
        pad_token_id=(
            tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id
        ),
        use_cache=True,
        stopping_criteria=stopping_criteria,
    )

    input_length = inputs["input_ids"].shape[1]
    output_length = outputs.shape[1] - input_length


    generated_ids = outputs[0, prompt_length:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return {
        "raw_output": raw_output,
        "output": parse_model_output(raw_output, output_type=output_type),
    }

# @torch.inference_mode()
# def predict(task_id="", input: str="", max_new_tokens: int = 1024, return_dict=True) -> dict:

#     task_id= 'test'
#     output_type = 'json'
#     messages = build_messages(task_id=task_id, input=input)

#     inputs = tokenizer.apply_chat_template(
#         messages,
#         tokenize=True,
#         add_generation_prompt=True,
#         return_tensors="pt",
#         return_dict=return_dict,
#     ).to(device)

#     # Qwen dùng <|im_end|> để kết thúc một lượt hội thoại
#     im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

#     stop_ids = [tokenizer.eos_token_id]
#     if im_end_id is not None and im_end_id not in stop_ids:
#         stop_ids.append(im_end_id)

#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         do_sample=False,
#         eos_token_id=stop_ids,
#         pad_token_id=(
#             tokenizer.pad_token_id
#             if tokenizer.pad_token_id is not None
#             else tokenizer.eos_token_id
#         ),
#         repetition_penalty=1.05,
#     )

#     input_length = inputs["input_ids"].shape[-1]
#     generated_ids = outputs[0, input_length:]

#     raw_output = tokenizer.decode(
#         generated_ids, skip_special_tokens=True
#     ).strip()


#     return {
#         "raw_output": raw_output,
#         "output": parse_model_output(raw_output, output_type=output_type)
#     }

In [26]:
import time

# đo thời gian đánh giá
def counter_predict_time(input: str, task_id: str = "LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION", max_new_tokens: int = 128):
    start = time.perf_counter()
    res = predict(
        task_id=task_id,
        input=input,
        return_dict=True,
        max_new_tokens=max_new_tokens
    )
    elapsed = time.perf_counter() - start

    print(f"Thời gian dự đoán cho input: {input} là {elapsed:.2f}s")
    print("Raw output:", res.get("raw_output"))
    print('output: ', res.get("output"))
    return elapsed, res

counter_predict_time(
    input="chương 1 mục 4 tiểu mục 7 trong bo luật dân su, điều 2 khoản 4, nói về chủ đề gì",
    task_id=TASK_ID,
    max_new_tokens=1024
)

Device: cuda:0
Input tokens: 248
Output tokens: 49
Thời gian dự đoán cho input: chương 1 mục 4 tiểu mục 7 trong bo luật dân su, điều 2 khoản 4, nói về chủ đề gì là 7.71s
Raw output: {"part": null, "chapter": "1", "section": "4", "subsection": "7", "article": "2", "clause": "4", "point": null, "document": "bo luật dân su"}
output:  {'part': None, 'chapter': '1', 'section': '4', 'subsection': '7', 'article': '2', 'clause': '4', 'point': None, 'document': 'bo luật dân su'}


(7.711983719000273,
 {'raw_output': '{"part": null, "chapter": "1", "section": "4", "subsection": "7", "article": "2", "clause": "4", "point": null, "document": "bo luật dân su"}',
  'output': {'part': None,
   'chapter': '1',
   'section': '4',
   'subsection': '7',
   'article': '2',
   'clause': '4',
   'point': None,
   'document': 'bo luật dân su'}})

In [9]:
from datasets import load_dataset


HF_DATASET_REPO = "tmtdev/ViLegal-QRHE-Dataset"
raw_test = load_dataset(HF_DATASET_REPO, split="test")

print("Số lượng mẫu test:", len(raw_test))
print("Ví dụ mẫu test:", raw_test[0])

README.md:   0%|          | 0.00/871 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 47.6kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8438 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1055 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1055 [00:00<?, ? examples/s]

Số lượng mẫu test: 1055
Ví dụ mẫu test: {'task_id': 'LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION', 'input': 'Nội dung Khoản 1 Điều 157 theo Bộ luật Dân sự 2015 là gì?', 'output': {'part': None, 'chapter': None, 'section': None, 'subsection': None, 'article': '157', 'clause': '1', 'point': None, 'document': 'Bộ luật Dân sự 2015'}}


In [29]:
from datetime import datetime
import pandas as pd

OUTPUT_DIR = Path('/content/drive/My Drive/legal-ai/models/v1/eval')

def save_eval_result(test_df: pd.DataFrame):
  time = datetime.now().strftime("%Y%m%d_%H%M%S")
  eval_file_name = f"test_evaluation_{TASK_ID}_{time}.csv"
  test_df.to_csv(
      os.path.join(OUTPUT_DIR, eval_file_name),
      index=False,
      encoding="utf-8-sig"
  )

  test_df.to_csv(
      eval_file_name,
      index=False,
      encoding="utf-8-sig"
  )

  print(f"Đã lưu eval result tại {eval_file_name}")

In [ ]:
test_list = list(raw_test)

# chạy 10 dòng

eval_result = []
batch_time = 0

for i in range(0, len(test_list)):
  if (i + 1) % 20 == 0:
    print(f"Dự đoán {i + 1}/{len(test_list)} mẫu")
    print(f"Thời gian dự đoán trung bình: {batch_time/20:.2f}s / mẫu")
    batch_time = 0
    save_eval_result(pd.DataFrame(eval_result))
  start = time.perf_counter()
  res = predict(
      task_id=test_list[i]['task_id'],
      input=test_list[i]['input'],
      return_dict=True,
      max_new_tokens=128
  )
  elapsed = time.perf_counter() - start
  batch_time += elapsed


  eval_result.append({
      'task_id': test_list[i]['task_id'],
      'input': test_list[i]['input'],
      'output': json.dumps(res.get("output"), ensure_ascii=False),
      'generated_output': json.dumps(res.get("output"), ensure_ascii=False),
      'raw_output': res.get("raw_output"),
      'response_time': elapsed
  })

In [ ]:
import pandas as pd

display(pd.DataFrame(eval_result))


In [61]:
@torch.inference_mode()
def predict_batch(
    items: list[dict], # list of {"task_id": str, "input": str}
    model,
    tokenizer,
    max_new_tokens: int = 1024,
    batch_size: int = 8
) -> list[dict]:
    """
    Xử lý batch hóa giảm thời gian từ 10s/sample xuống ~0.5s/sample với GPU tầm trung.
    """
    results = []

    # Thiết lập pad token bên trái cho Generation Batching
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    stop_ids = [tokenizer.eos_token_id]
    if im_end_id is not None and im_end_id not in stop_ids:
        stop_ids.append(im_end_id)


    for i in range(0, len(items), batch_size):
        batch_items = items[i : i + batch_size]
        batch_prompts = []

        start = time.perf_counter()

        for item in batch_items:
            messages = build_messages(task_id=item["task_id"], input=item["input"])
            text_prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            batch_prompts.append(text_prompt)

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=stop_ids,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

        # Cut bỏ phần prompt input để lấy generated token
        input_len = inputs["input_ids"].shape[1]
        generated_ids = outputs[:, input_len:]

        decoded_outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        for idx, raw_out in enumerate(decoded_outputs):
            task_id = batch_items[idx]["task_id"]
            out_type = TASK_REGISTRY[task_id].get("output_type", "json") if isinstance(TASK_REGISTRY[task_id], dict) else "str"

            results.append({
                "raw_output": raw_out.strip(),
                "output": parse_model_output(raw_out.strip(), output_type=out_type)
            })

        elapsed = time.perf_counter() - start

        print(f"Dự đoán {i + len(batch_items)}/{len(items)} mẫu")
        print(f"Thời gian dự đoán trung bình: {elapsed/len(batch_items):.2f}s / mẫu")


    return results

results = predict_batch(
    items = test_list, # list of {"task_id": str, "input": str}
    model=model,
    tokenizer=tokenizer,
    max_new_tokens = 128,
    batch_size = 32
)

Dự đoán 32/1055 mẫu
Thời gian dự đoán trung bình: 0.38s / mẫu
Dự đoán 64/1055 mẫu
Thời gian dự đoán trung bình: 0.38s / mẫu
Dự đoán 96/1055 mẫu
Thời gian dự đoán trung bình: 0.38s / mẫu
Dự đoán 128/1055 mẫu
Thời gian dự đoán trung bình: 0.38s / mẫu
Dự đoán 160/1055 mẫu
Thời gian dự đoán trung bình: 0.39s / mẫu
Dự đoán 192/1055 mẫu
Thời gian dự đoán trung bình: 0.37s / mẫu
Dự đoán 224/1055 mẫu
Thời gian dự đoán trung bình: 0.37s / mẫu
Dự đoán 256/1055 mẫu
Thời gian dự đoán trung bình: 0.38s / mẫu
Dự đoán 288/1055 mẫu
Thời gian dự đoán trung bình: 0.37s / mẫu
Dự đoán 320/1055 mẫu
Thời gian dự đoán trung bình: 0.38s / mẫu
Dự đoán 352/1055 mẫu
Thời gian dự đoán trung bình: 0.37s / mẫu
Dự đoán 384/1055 mẫu
Thời gian dự đoán trung bình: 0.39s / mẫu
Dự đoán 416/1055 mẫu
Thời gian dự đoán trung bình: 0.37s / mẫu
Dự đoán 448/1055 mẫu
Thời gian dự đoán trung bình: 0.37s / mẫu
Dự đoán 480/1055 mẫu
Thời gian dự đoán trung bình: 0.40s / mẫu
Dự đoán 512/1055 mẫu
Thời gian dự đoán trung bình: 0.39s /

In [64]:
eval_result = []
for i in range(len(test_list)):
  generated_output = results[i]['output']
  raw_output = results[i]['raw_output']

  expected_output = test_list[i]['output']

  valid_json = generated_output is not None

  correct_keys = (
      valid_json
      and set(generated_output.keys()) == {
          "document", "part", "chapter", "section", "subsection",
          "article", "clause", "point"
      }
  )

  output_ok = (
      correct_keys
      and isinstance(generated_output, type(expected_output))
  )

  eval_result.append({
      'task_id': test_list[i]['task_id'],
      "input": test_list[i]['input'],
      "output": json.dumps(expected_output, ensure_ascii=False),
      "generated_output": json.dumps(generated_output, ensure_ascii=False) if valid_json else None,
      "valid_json": valid_json,
      "correct_keys": correct_keys,
      "output_format_ok": output_ok,
      "output_exact_match": (
          generated_output == expected_output
          if correct_keys else False
      ),
      "raw_output": raw_output
  })





In [65]:
print(f"Đã đánh giá {len(eval_result)} mẫu")
# Lưu đánh giá
save_eval_result(pd.DataFrame(eval_result))



Đã đánh giá 1055 mẫu
Đã lưu eval result tại test_evaluation_LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION_20261015_181942.csv


In [66]:
df = pd.DataFrame(eval_result)

# Tổng hợp các metric
metrics = [
    "valid_json",
    "correct_keys",
    "output_format_ok",
    "output_exact_match",
]

summary = pd.DataFrame({
    "metric": metrics,
    "score_percent": [
        100 * df[m].mean() for m in metrics
    ]
})

print(summary.to_string(index=False))

# Xem những mẫu model dự đoán sai
errors = df[~df["output_exact_match"]]

display(errors.head(10))


            metric  score_percent
        valid_json     100.000000
      correct_keys     100.000000
  output_format_ok     100.000000
output_exact_match      99.905213


,task_id,input,output,generated_output,valid_json,correct_keys,output_format_ok,output_exact_match,raw_output
648,LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION,"Theo Điều 624 và Điều 626 Bộ luật dân sự 2015,...","{""part"": null, ""chapter"": null, ""section"": nul...","{""part"": null, ""chapter"": null, ""section"": nul...",True,True,True,False,"{""part"": null, ""chapter"": null, ""section"": nul..."
